# Building JNode Trees

The [`fn:jtree()`]({docs}/functions/fn/jtree) function is the gateway from maps and arrays to navigable JNode trees. It accepts any map or array and returns a JNode root.

## From Maps

A map becomes an **object node** whose children are **member nodes**, one per key-value pair:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(map { "city": "Boston", "state": "MA", "zip": "02101" })
return (
    "Root kind: " || $tree instance of object-node(),
    "Children: " || count($tree/child::*),
    "Keys: " || string-join(
        for $c in $tree/child::* return fn:jkey($c), ", "
    )
)

## From Arrays

An array becomes an **array node** whose children are its members, each with a positional index:

In [ ]:
xquery version "4.0";

let $tree := fn:jtree(array { "red", "green", "blue" })
return (
    "Root kind: " || $tree instance of array-node(),
    "Members: " || count($tree/child::*),
    "Positions: " || string-join(
        for $c in $tree/child::* return string(fn:jposition($c)), ", "
    )
)

## Nested Structures

JNode trees mirror the full depth of the original data. Nested maps and arrays create nested object and array nodes:

In [ ]:
xquery version "4.0";

let $data := map {
    "library": map {
        "name": "City Library",
        "books": array {
            map { "title": "Hamlet", "author": "Shakespeare" },
            map { "title": "Ulysses", "author": "Joyce" }
        }
    }
}
let $tree := fn:jtree($data)
return map {
    "total nodes": count($tree/descendant::*),
    "depth-2 keys": string-join(
        for $c in $tree/child::*/child::* return fn:jkey($c), ", "
    )
}

## From Parsed JSON

A common workflow: parse a JSON string, convert to a JNode tree, then navigate:

In [ ]:
xquery version "4.0";

let $json := '{
    "name": "eXist-db",
    "version": 7,
    "features": ["XQuery", "XSLT", "Full Text", "JNodes"]
}'
let $tree := fn:jtree(parse-json($json))
return (
    fn:jvalue($tree/name) || " v" || fn:jvalue($tree/version),
    "Features: " || string-join(
        for $f in $tree/features/child::* return fn:jvalue($f), ", "
    )
)

Notice how `$tree/name` navigates to the child with key `"name"` — just like `$element/child` navigates to child elements in XML. This is the power of JNodes: JSON navigation uses the same syntax as XML navigation.

## JNode Functions

Here is a summary of the JNode-specific functions:

| Function | Purpose |
|----------|---------|
| `fn:jtree($value)` | Convert a map or array to a JNode tree |
| `fn:jkey($node)` | Get the key of a member node (from a map entry) |
| `fn:jvalue($node)` | Get the underlying value of a JNode |
| `fn:jposition($node)` | Get the 1-based position of a node among its siblings |
| `fn:jchildren($node)` | Get the children of a node (same as `child::*`) |
| `fn:jparent($node)` | Get the parent of a node (same as `parent::*`) |